# Day 1 — First inference + token inspector

Goal: load a tiny LLM, generate text, and inspect token probabilities at each position.
By the end you should *feel* what a model is doing token by token, not just see strings come out.

Models that fit on a Mac or modest CPU box:
- `Qwen/Qwen2-0.5B-Instruct` — recommended
- `TinyLlama/TinyLlama-1.1B-Chat-v1.0`
- `HuggingFaceTB/SmolLM2-360M-Instruct` — smallest, fastest

Pick one, replace `MODEL_NAME` below, run all cells, commit.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2-0.5B-Instruct"
DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", DEVICE)

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32).to(DEVICE)
model.eval()
print("params:", sum(p.numel() for p in model.parameters()) / 1e6, "M")

## 1. Generate

In [ ]:
prompt = "In one short sentence, what is the capital of France?"

messages = [{"role": "user", "content": prompt}]
input_ids = tok.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True).to(DEVICE)

with torch.no_grad():
    out = model.generate(
        input_ids,
        max_new_tokens=40,
        do_sample=False,
        pad_token_id=tok.eos_token_id,
        return_dict_in_generate=True,
        output_scores=True,
    )

generated = tok.decode(out.sequences[0][input_ids.shape[1]:], skip_special_tokens=True)
print("--- completion ---")
print(generated)

## 2. Token inspector — print top-k logits at each generated position

In [ ]:
import torch.nn.functional as F
import pandas as pd

TOP_K = 5
rows = []
gen_tokens = out.sequences[0][input_ids.shape[1]:]

for step, (tok_id, scores) in enumerate(zip(gen_tokens, out.scores)):
    probs = F.softmax(scores[0], dim=-1)
    top_probs, top_ids = torch.topk(probs, TOP_K)
    entropy = -(probs * probs.clamp_min(1e-12).log()).sum().item()
    rows.append({
        "step": step,
        "chosen_token": repr(tok.decode([tok_id.item()])),
        "chosen_prob": probs[tok_id].item(),
        "entropy_nats": entropy,
        "top_k": [(repr(tok.decode([i.item()])), round(p.item(), 3))
                  for p, i in zip(top_probs, top_ids)],
    })

df = pd.DataFrame(rows)
df

## 3. Your turn — answer these questions in the cell below

1. Which positions had the *lowest entropy* (most confident)? Why those?
2. Where the model was *least confident*, what were the top alternative tokens? Did they share a theme?
3. Try changing the prompt to something ambiguous (e.g. *"The next word is"*) and re-run. How does the entropy distribution change?

Treat this notebook as a thinking tool. Save your observations as markdown cells — they become blog material.

*observations: ...*